# AutoRedTeam — Fine-tuning Llama 3.1 8B on CVE Analysis

**Runtime:** T4 GPU (free Colab) — ~45 min for 500 examples, 3 epochs

**Steps:**
1. Install Unsloth
2. Upload your `finetune_dataset.jsonl` when prompted (Cell 5)
3. Run all cells in order
4. LoRA adapter saved to Google Drive

In [ ]:
# Cell 1 — Install Unsloth (~3 min)
!pip install unsloth -q
!pip install --upgrade --no-cache-dir unsloth -q

In [ ]:
# Cell 2 — Verify GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 3 — Load base model with 4-bit QLoRA
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/Meta-Llama-3.1-8B-Instruct',
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)
print('Model loaded.')

In [ ]:
# Cell 4 — Attach LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
# Cell 5 — Upload dataset
from google.colab import files
print('Upload finetune_dataset.jsonl from your Mac:')
uploaded = files.upload()

In [ ]:
# Cell 6 — Load and format dataset
import json
from datasets import Dataset

ALPACA_PROMPT = '''Below is an instruction that describes a task, paired with input. Write a response.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}'''

EOS = tokenizer.eos_token

with open('finetune_dataset.jsonl') as f:
    raw = [json.loads(l) for l in f]

def format_example(ex):
    text = ALPACA_PROMPT.format(
        instruction=ex['instruction'],
        input=ex['input'],
        output=ex['output'],
    ) + EOS
    return {'text': text}

dataset = Dataset.from_list([format_example(ex) for ex in raw])
print(f'Dataset: {len(dataset)} examples')
print(dataset[0]['text'][:400])

In [ ]:
# Cell 7 — Train (~45 min on T4)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = 'text',
    max_seq_length     = MAX_SEQ_LEN,
    dataset_num_proc   = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs            = 3,
        warmup_steps                = 10,
        learning_rate               = 2e-4,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 10,
        optim                       = 'adamw_8bit',
        weight_decay                = 0.01,
        lr_scheduler_type           = 'linear',
        output_dir                  = 'autoredteam_checkpoints',
        report_to                   = 'none',
    ),
)
print('Starting training...')
trainer.train()

In [ ]:
# Cell 8 — Quick inference test
FastLanguageModel.for_inference(model)

test_cve = '''CVE ID: CVE-2021-44228
Severity: CRITICAL (CVSS 10.0)
Published: 2021-12-10
CWE: CWE-917
Description: Apache Log4j2 JNDI features do not protect against attacker controlled LDAP endpoints.
An attacker who can control log messages can execute arbitrary code via LDAP servers.'''

prompt = ALPACA_PROMPT.format(
    instruction='You are an expert penetration tester. Analyze the following CVE and provide a structured security assessment.',
    input=test_cve,
    output='',
)
inputs  = tokenizer([prompt], return_tensors='pt').to('cuda')
outputs = model.generate(**inputs, max_new_tokens=400, use_cache=True)
print(tokenizer.batch_decode(outputs)[0].split('### Response:')[1])

In [ ]:
# Cell 9 — Save LoRA adapter to Google Drive
from google.colab import drive
drive.mount('/content/drive')

SAVE_PATH = '/content/drive/MyDrive/autoredteam_lora'
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'Saved to Google Drive: {SAVE_PATH}')
print('Download to: autoredteam/models/autoredteam_lora/')